In [1]:
import cv2 as cv
import time
from yolo_detector import YoloDetector
from tracker import Tracker

In [2]:
MODEL_PATH = "models/yolo11m.pt"
VIDEO_PATH = "traffic.mp4"

def main():
    # Initialize YOLO detector and Deep SORT tracker
    detector = YoloDetector(model_path=MODEL_PATH, confidence=0.2)
    tracker = Tracker()

    # Open the video file
    cap = cv.VideoCapture(VIDEO_PATH)

    # Get the frames per second (fps) and frame size of the video
    fps = int(cap.get(cv.CAP_PROP_FPS))
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))

    # Define the codec and create VideoWriter object for output
    fourcc = cv.VideoWriter_fourcc(*'mp4v')
    out = cv.VideoWriter('traffic-Output.mp4', fourcc, fps, (width, height))

    if not cap.isOpened():
        print("Error: Unable to open video file.")
        exit()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        start_time = time.perf_counter()
        # Perform detection on the current frame
        detections = detector.detect(frame)
        # Update tracker based on detections
        tracking_ids, boxes = tracker.track(detections, frame)

        for detection, tracking_id, bounding_box in zip(detections, tracking_ids, boxes):
            (bbox, class_number, conf) = detection
            x1, y1, w, h = bbox
            # Draw the bounding box
            cv.rectangle(frame, (int(bounding_box[0]), int(bounding_box[1])), 
                          (int(bounding_box[2]), int(bounding_box[3])), (0, 0, 255), 2)
            # Display the tracking ID and class name
            class_name = detector.model.names[class_number]
            cv.putText(frame, f"{class_name} ID: {str(tracking_id)}", (int(bounding_box[0]), int(bounding_box[1] - 10)), 
                        cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        end_time = time.perf_counter()
        fps = 1 / (end_time - start_time)
        print(f"Current fps: {fps}")

        # Display the processed frame
        cv.imshow("Frame", frame)
        
        # Write the resulting frame to output video file
        out.write(frame)

        key = cv.waitKey(1) & 0xFF
        if key == ord("q") or key == 27:
            break

    # Cleanup
    cap.release()
    out.release()
    cv.destroyAllWindows()

if __name__ == "__main__":
    main()



0: 384x640 2 persons, 22 cars, 3 buss, 2 trucks, 1 traffic light, 76.0ms
Speed: 3.4ms preprocess, 76.0ms inference, 144.6ms postprocess per image at shape (1, 3, 384, 640)
Current fps: 1.0333127325546074

0: 384x640 2 persons, 23 cars, 2 buss, 1 truck, 1 traffic light, 32.1ms
Speed: 1.3ms preprocess, 32.1ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)
Current fps: 4.603107229020161

0: 384x640 2 persons, 23 cars, 1 bus, 1 truck, 1 traffic light, 32.2ms
Speed: 1.6ms preprocess, 32.2ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Current fps: 6.776011847413089

0: 384x640 2 persons, 23 cars, 1 bus, 1 truck, 1 traffic light, 14.1ms
Speed: 1.4ms preprocess, 14.1ms inference, 12.3ms postprocess per image at shape (1, 3, 384, 640)
Current fps: 9.829345749597485

0: 384x640 2 persons, 23 cars, 1 bus, 1 truck, 1 traffic light, 28.7ms
Speed: 1.3ms preprocess, 28.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
Current fps: 10.68224645